In [85]:

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName('big-data-management-assignment-1')
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .enableHiveSupport()   # persist tables to local Hive metastore
    .getOrCreate()
)

print(spark.version)

4.1.0


In [86]:
# Step 1: Incremental Ingestion - Schema Definition
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType, TimestampType
import json
import os
from pathlib import Path
from datetime import datetime

# Define explicit schema for NYC Yellow Taxi Trip Records
# Based on TLC data dictionary: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# Note: Parquet files store integer columns as INT64, so we use LongType instead of IntegerType
yellow_taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

print(f"Schema defined with {len(yellow_taxi_schema.fields)} fields")

Schema defined with 19 fields


In [87]:
# Manifest Handling Functions

MANIFEST_PATH = "state/manifest.json"

def load_manifest():
    """Load the manifest of already processed files."""
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, 'r') as f:
            return json.load(f)
    return {"processed_files": [], "last_updated": None}

def save_manifest(manifest):
    """Save the manifest to disk."""
    os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)
    
    # Add timestamp
    manifest["last_updated"] = datetime.now().isoformat()
    
    # Write to file with pretty formatting
    with open(MANIFEST_PATH, 'w') as f:
        json.dump(manifest, indent=2, fp=f)
    print(f"Manifest saved: {len(manifest['processed_files'])} files tracked")

def get_new_files(input_dir, manifest):
    """Identify files in input_dir that haven't been processed yet."""
    processed_files = set(manifest.get("processed_files", []))
    
    # List all parquet files in input directory
    input_path = Path(input_dir)
    all_files = [str(f.relative_to(input_path.parent)) for f in input_path.glob("yellow_tripdata_*.parquet")]
    
    # Filter to only new files
    new_files = [f for f in all_files if f not in processed_files]
    
    return sorted(new_files)

# Test manifest functions
manifest = load_manifest()
print(f"Current manifest: {len(manifest.get('processed_files', []))} files already processed")

Current manifest: 0 files already processed


In [88]:
# Incremental File Reading Function

def read_incremental_data(input_dir, manifest, schema):
    """
    Read only new parquet files that haven't been processed yet.
    
    Args:
        input_dir: Directory containing input parquet files
        manifest: Dict containing already processed files
        schema: Explicit StructType schema for reading
    
    Returns:
        tuple: (DataFrame with new data, updated manifest dict)
               Returns (None, manifest) if no new files found
    """
    new_files = get_new_files(input_dir, manifest)
    
    if not new_files:
        print("No new files to process")
        return None, manifest
    
    print(f"Found {len(new_files)} new file(s) to process:")
    for f in new_files:
        print(f"   • {f}")
    
    # Read all new files with explicit schema
    file_paths = [os.path.join(input_dir, f.split('/')[-1]) for f in new_files]
    
    # Configure Spark for optimized reads
    # maxPartitionBytes controls split size - 128MB 
    spark.conf.set("spark.sql.files.maxPartitionBytes", "128MB")
    
    # Read parquet files with explicit schema
    df = spark.read.schema(schema).parquet(*file_paths)
    
    # Add metadata columns for lineage tracking
    # - source_file: which parquet file this row came from
    # - ingested_at: when this batch was processed
    df = (df
          .withColumn("source_file", F.input_file_name())
          .withColumn("ingested_at", F.lit(datetime.now().isoformat()))
    )
    
    # Get row counts per file
    row_count = df.count()
    print(f"Read {row_count:,} rows from {len(new_files)} file(s)")
    
    # Update manifest
    manifest["processed_files"].extend(new_files)
    
    return df, manifest


In [89]:
# Execute Incremental Ingestion

# Configuration
INPUT_DIR = "data/inbox"

# Load manifest
manifest = load_manifest()

# Read only new files
raw_df, manifest = read_incremental_data(INPUT_DIR, manifest, yellow_taxi_schema)

if raw_df is not None:
    # Show sample of data
    print("\nSample of ingested data:")
    raw_df.select(
        "tpep_pickup_datetime", 
        "PULocationID", 
        "DOLocationID", 
        "passenger_count", 
        "trip_distance",
        "source_file",
        "ingested_at"
    ).show(5, truncate=80)
    
    # Show data summary
    print("\nData Summary:")
    print(f"   Total rows: {raw_df.count():,}")
    print(f"   Columns: {len(raw_df.columns)}")
    print(f"   Partitions: {raw_df.rdd.getNumPartitions()}")
    
    print("\nStep 1 Complete - Data ready for transformation")
else:
    print("\nNo new data to process")

Found 2 new file(s) to process:
   • inbox/yellow_tripdata_2025-01.parquet
   • inbox/yellow_tripdata_2025-02.parquet
Read 7,052,769 rows from 2 file(s)

Sample of ingested data:
+--------------------+------------+------------+---------------+-------------+-------------------------------------------------------------------+--------------------------+
|tpep_pickup_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|                                                        source_file|               ingested_at|
+--------------------+------------+------------+---------------+-------------+-------------------------------------------------------------------+--------------------------+
| 2025-01-01 00:18:38|         229|         237|              1|          1.6|file:///home/jovyan/work/data/inbox/yellow_tripdata_2025-01.parquet|2026-03-07T23:19:49.600831|
| 2025-01-01 00:32:40|         236|         237|              1|          0.5|file:///home/jovyan/work/data/inbox/yellow_trip

In [90]:
# Inspect Schema and Data Quality

if raw_df is not None:
    
    # Show schema to verify types
    print("\nSchema (verify types are correct):")
    raw_df.printSchema()
    
    # Check for nulls in critical columns
    print("\nNull count in critical columns:")
    raw_df.select([
        F.sum(F.when(F.col("tpep_pickup_datetime").isNull(), 1).otherwise(0)).alias("null_pickup_dt"),
        F.sum(F.when(F.col("PULocationID").isNull(), 1).otherwise(0)).alias("null_pickup_loc"),
        F.sum(F.when(F.col("DOLocationID").isNull(), 1).otherwise(0)).alias("null_dropoff_loc"),
        F.sum(F.when(F.col("trip_distance").isNull(), 1).otherwise(0)).alias("null_distance")
    ]).show()
    
    # Basic statistics on numeric columns
    print("\nBasic statistics:")
    raw_df.select("trip_distance", "passenger_count", "total_amount").describe().show()
    
else:
    print("No data loaded - skipping quality checks")


Schema (verify types are correct):
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingested_at: string (nullable = false)


Null count in cri

In [91]:
# ============================================================
# Optional: Test Incremental Behavior (Idempotency Check)
# ============================================================
# This cell verifies that re-running won't duplicate data.
# Expected: After first run, this shows 0 new files.
# ============================================================

# Load current manifest state
test_manifest = load_manifest()

print(f"\nFiles currently in manifest: {len(test_manifest.get('processed_files', []))}")
if test_manifest.get('processed_files'):
    for idx, file in enumerate(test_manifest.get('processed_files', []), 1):
        print(f"   {idx}.  {file}")
else:
    print("   (empty - no files processed yet)")

# Check for new files
test_new = get_new_files(INPUT_DIR, test_manifest)

print(f"\nNew files detected: {len(test_new)}")
if test_new:
    print(" Files waiting to be processed:")
    for f in test_new:
        print(f"      • {f}")
    print("\n  Run the execution cell above to process these files")
else:
    print("  No new files found")
    print("  Incremental logic working correctly - no duplicates!")
    print("\n To test incremental behavior:")
    print("      1. Add a new yellow_tripdata_YYYY-MM.parquet to data/input/")
    print("      2. Re-run the execution cell above")
    print("      3. Only the new file will be processed")



Files currently in manifest: 0
   (empty - no files processed yet)

New files detected: 2
 Files waiting to be processed:
      • inbox/yellow_tripdata_2025-01.parquet
      • inbox/yellow_tripdata_2025-02.parquet

  Run the execution cell above to process these files


In [92]:
# Step 2: Transformations - Cleaning and Deduplication

if raw_df is not None:
    raw_count = raw_df.count()
    print(f"Raw row count: {raw_count:,}")

    # Checking for null or critical columns

    cleaned_df = (
        raw_df
        .filter(F.col("tpep_pickup_datetime").isNotNull())
        .filter(F.col("tpep_dropoff_datetime").isNotNull())
        .filter(F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime"))
        .filter(F.col("trip_distance").isNotNull())
        .filter(F.col("trip_distance") > 0)
        .filter(F.col("passenger_count").isNotNull())
        .filter(F.col("passenger_count") > 0)
        .filter(F.col("passenger_count") <= 6)
        .filter(F.col("PULocationID").isNotNull())
        .filter(F.col("DOLocationID").isNotNull())
        .filter(F.col("fare_amount") >= 0)
    )

    cleaned_count = cleaned_df.count()
    print(f"After cleaning: {cleaned_count:,} rows ({raw_count - cleaned_count:,} removed)")

    # This combination uniquely identifies a trip
    dedup_df = cleaned_df.dropDuplicates([
        "VendorID",
        "tpep_pickup_datetime",
        "PULocationID",
        "DOLocationID"
    ])

    dedup_count = dedup_df.count()
    print(f"After deduplication: {dedup_count:,} rows ({cleaned_count - dedup_count:,} removed)")

else:
    print("No new data to process - skipping transformations")

Raw row count: 7,052,769
After cleaning: 5,473,994 rows (1,578,775 removed)
After deduplication: 5,470,088 rows (3,906 removed)


In [93]:
# Step 3: Enrichment - Join with taxi zone lookup table

if raw_df is not None:

    # Read the zone lookup table
    zone_df = spark.read.parquet("data/inbox/taxi_zone_lookup.parquet")

    print("Zone lookup schema:")
    zone_df.printSchema()
    print(f"Zone lookup row count: {zone_df.count()}")

    # Rename columns before joining to avoid ambiguity
    pickup_zone_df = zone_df.select(
        F.col("LocationID").alias("pickup_location_id"),
        F.col("Zone").alias("pickup_zone"),
        F.col("Borough").alias("pickup_borough")
    )

    dropoff_zone_df = zone_df.select(
        F.col("LocationID").alias("dropoff_location_id"),
        F.col("Zone").alias("dropoff_zone"),
        F.col("Borough").alias("dropoff_borough")
    )

    # Join pickup zone
    enriched_df = dedup_df.join(
        F.broadcast(pickup_zone_df),
        dedup_df.PULocationID == pickup_zone_df.pickup_location_id,
        "left"
    ).drop("pickup_location_id")

    # Join dropoff zone
    enriched_df = enriched_df.join(
        F.broadcast(dropoff_zone_df),
        enriched_df.DOLocationID == dropoff_zone_df.dropoff_location_id,
        "left"
    ).drop("dropoff_location_id")

    print("Enrichment complete")
    enriched_df.select(
        "tpep_pickup_datetime",
        "PULocationID",
        "pickup_zone",
        "DOLocationID",
        "dropoff_zone"
    ).show(5, truncate=50)

else:
    print("No new data to process - skipping enrichment")

Zone lookup schema:
root
 |-- LocationID: long (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

Zone lookup row count: 265
Enrichment complete
+--------------------+------------+------------------------+------------+---------------+
|tpep_pickup_datetime|PULocationID|             pickup_zone|DOLocationID|   dropoff_zone|
+--------------------+------------+------------------------+------------+---------------+
| 2025-01-01 00:59:34|          79|            East Village|         249|   West Village|
| 2025-01-01 00:32:01|         234|                Union Sq|          79|   East Village|
| 2025-01-01 00:18:33|          79|            East Village|         140|Lenox Hill East|
| 2025-01-01 00:52:00|          87|Financial District North|         262| Yorkville East|
| 2025-01-01 00:14:21|         238|   Upper West Side North|          24|   Bloomingdale|
+--------------------+------------+-------------

In [94]:
# Step 4: Output - Add derived columns and write partitioned parquet

if raw_df is not None:

    # Add derived columns
    output_df = (
        enriched_df
        .withColumn(
            "trip_duration_minutes",
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
        )
        .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
        .withColumn("pickup_year_month", F.date_format("tpep_pickup_datetime", "yyyy-MM"))
    )

    # Select only the required output fields
    output_df = output_df.select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "pickup_zone",
        "pickup_borough",
        "dropoff_zone",
        "dropoff_borough",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "trip_duration_minutes",
        "pickup_date",
        "pickup_year_month",
        "source_file",
        "ingested_at"
    )

    OUTPUT_DIR = "data/outbox/trips_enriched.parquet"
    try:
        # Dynamic partition overwrite ensures only new partitions are written
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

        output_df.write \
            .mode("overwrite") \
            .partitionBy("pickup_year_month") \
            .parquet(OUTPUT_DIR)

        final_count = output_df.count()
        print(f"Final output row count: {final_count:,}")
        print(f"Written to {OUTPUT_DIR} partitioned by pickup_year_month")

        # Save manifest only after successful write
        save_manifest(manifest)
        print("Manifest updated")

    except Exception as e:
        print(f"Error occurred while writing output: {e}")

else:
    print("No new data to process - skipping output write")

Final output row count: 5,470,088
Written to data/outbox/trips_enriched.parquet partitioned by pickup_year_month
Manifest saved: 2 files tracked
Manifest updated


In [95]:
# Display sample data in a formatted way
from IPython.display import display
import pandas as pd

sample_df = output_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "pickup_zone",
    "pickup_borough",
    "dropoff_zone",
    "dropoff_borough",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "trip_duration_minutes",
    "pickup_date",
    "pickup_year_month",
    "source_file",
    "ingested_at"
).limit(5).toPandas()

# Pretty print with pandas styling
print("NYC TAXI TRIP DATA - SAMPLE OUTPUT ")
display(sample_df)
print(f"\nTotal Records Processed: {output_df.count():,}")

NYC TAXI TRIP DATA - SAMPLE OUTPUT 


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,pickup_zone,pickup_borough,dropoff_zone,dropoff_borough,passenger_count,trip_distance,fare_amount,total_amount,trip_duration_minutes,pickup_date,pickup_year_month,source_file,ingested_at
0,2025-01-01 00:01:53,2025-01-01 00:14:42,132,63,JFK Airport,Queens,Cypress Hills,Brooklyn,1,6.4,26.1,37.2,12.816667,2025-01-01,2025-01,file:///home/jovyan/work/data/inbox/yellow_tri...,2026-03-07T23:19:49.600831
1,2025-01-01 00:02:28,2025-01-01 00:13:11,261,232,World Trade Center,Manhattan,Two Bridges/Seward Park,Manhattan,1,2.2,12.8,17.8,10.716667,2025-01-01,2025-01,file:///home/jovyan/work/data/inbox/yellow_tri...,2026-03-07T23:19:49.600831
2,2025-01-01 00:06:19,2025-01-01 00:23:28,162,166,Midtown East,Manhattan,Morningside Heights,Manhattan,1,4.4,21.2,30.2,17.150000,2025-01-01,2025-01,file:///home/jovyan/work/data/inbox/yellow_tri...,2026-03-07T23:19:49.600831
3,2025-01-01 00:06:44,2025-01-01 00:29:44,114,225,Greenwich Village South,Manhattan,Stuyvesant Heights,Brooklyn,1,4.5,23.3,32.3,23.000000,2025-01-01,2025-01,file:///home/jovyan/work/data/inbox/yellow_tri...,2026-03-07T23:19:49.600831
4,2025-01-01 00:08:51,2025-01-01 00:17:04,164,114,Midtown South,Manhattan,Greenwich Village South,Manhattan,2,1.8,10.7,18.8,8.216667,2025-01-01,2025-01,file:///home/jovyan/work/data/inbox/yellow_tri...,2026-03-07T23:19:49.600831



Total Records Processed: 5,470,088
